In [2]:
import numpy as np

In [3]:
v = np.array([3,4])

In [4]:
print(v)

[3 4]


In [5]:
print(type(v))

<class 'numpy.ndarray'>


In [6]:
w = np.array([1, 2])

# Vector Addition

In [7]:
print(v + w)

[4 6]


# Scalar multiplication

In [8]:
print(w)

[1 2]


In [9]:
print(v*2)

[6 8]


# dot product

In [10]:
dot = np.dot(v, w)

In [11]:
print(dot)

11


# vector magnitude (length)

In [12]:
length = np.linalg.norm(v)

In [13]:
print(length)

5.0


In [14]:
A = np.array([[1, 2], [3, 4]])

In [15]:
print(A)

[[1 2]
 [3 4]]


In [16]:
print(A.shape)

(2, 2)


# matrix multiplication

In [17]:
B = np.array([[5, 6], [7, 8]])

In [18]:
result = np.matmul(A, B)

In [19]:
print(result)

[[19 22]
 [43 50]]


#### matrix multiplication is just many dot products organized into a grid.

#### When a neural network processes a batch of inputs, it's literally doing inputs @ weights — one matrix multiplication, applying the same dot-product logic to every data point in the batch simultaneously. That's why GPUs are good at ML — they're built to do thousands of these multiplications in parallel.

### identity and transpose matrix

In [20]:
I = np.eye(2)

In [21]:
print(I)

[[1. 0.]
 [0. 1.]]


In [22]:
print(A.T)

[[1 3]
 [2 4]]


##### np.eye(2) gives the identity matrix — [[1,0],[0,1]]. Think of it like the number "1" but for matrices: multiply anything by it and you get the same thing back. It shows up in ML for things like resetting weights or solving systems of equations.
A.T is the transpose — rows become columns. Your [[1,2],[3,4]] became [[1,3],[2,4]]. This is everywhere in ML code — you'll see .T constantly when reshaping data to match what a model expects, or in the math behind backpropagation.
That's a genuinely solid first session: vectors, addition, scalar multiplication, dot product, magnitude, matrices, matrix multiplication, identity matrix, transpose — with actual intuition behind every one, not just syntax memorized.

#### EigenVectors and Eigenvalues

Eigenvalues and Eigenvectors — genuinely one of the most important concepts for ML (PCA, dimensionality reduction, understanding what a model "pays attention to").

When you multiply a matrix by a vector, the vector usually changes direction. An eigenvector is special — it's a vector that, when multiplied by the matrix, doesn't change direction, only gets scaled (stretched or shrunk). That scaling factor is the eigenvalue.

In [23]:
A = np.array([[4, 1], [2, 3]])

In [24]:
eigenvalues, eigenvectors = np.linalg.eig(A)

In [25]:
print(eigenvalues)

[5. 2.]


In [26]:
print(eigenvectors)

[[ 0.70710678 -0.4472136 ]
 [ 0.70710678  0.89442719]]


In [27]:
v1 = eigenvectors[:, 0]

In [28]:
print(A @ v1)

[3.53553391 3.53553391]


In [29]:
print(5 * v1)

[3.53553391 3.53553391]


They match — that's the proof. A @ v1 and 5 * v1 give the same result, confirming v1 is genuinely an eigenvector with eigenvalue 5. The matrix stretched it by exactly 5x without rotating it at all.

This is the math behind PCA (Principal Component Analysis), one of the most common dimensionality reduction techniques. When you have data with hundreds of features, PCA finds the eigenvectors of the data's covariance matrix — those eigenvectors point in the directions where your data varies the most. The eigenvalues tell you how much variance is in each direction. You keep the top few and drop the rest, compressing your data while losing minimal information.

You'll use this constantly without even calling np.linalg.eig() directly — sklearn.decomposition.PCA does this under the hood.

In [30]:
B = np.array([[2, 0], [0, 3]])

In [31]:
eigvals, eigvecs = np.linalg.eig(B)

In [32]:
print(eigvals)

[2. 3.]


In [33]:
print(eigvecs)

[[1. 0.]
 [0. 1.]]


This is the key insight: for a diagonal matrix, the eigenvalues are just the diagonal entries themselves, and the eigenvectors are the standard axes. No rotation happens at all — the matrix just stretches x-direction by 2 and y-direction by 3, independently.

This is actually why eigenvectors matter so much: they reveal the matrix's "natural directions" — the axes along which it behaves simply (just scaling, no mixing). For a messy, non-diagonal matrix, the eigenvectors tell you which directions in space behave simply, even though the matrix looks complicated in the standard x/y view.

##### PCA (Principal Component Analysis)

In [34]:
np.random.seed(42)

In [35]:
x = np.random.randn(100)

In [36]:
y = 2*x + np.random.randn(100)*0.5

In [37]:
data = np.column_stack((x, y))

In [38]:
print(data.shape)

(100, 2)


In [39]:
print(data[:5])

[[ 0.49671415  0.28574293]
 [-0.1382643  -0.48685126]
 [ 0.64768854  1.12401982]
 [ 1.52302986  2.64492108]
 [-0.23415337 -0.54894961]]


This simulates 100 data points where y is roughly proportional to x (correlated data, like height and weight might be).

Now let's find the directions of maximum variance — the actual PCA step. First we compute the covariance matrix, then find its eigenvectors:

In [40]:
cov_matrix = np.cov(data.T)

In [41]:
print(cov_matrix)

[[0.82476989 1.59046269]
 [1.59046269 3.29014231]]


In [42]:
eigvals, eigvecs = np.linalg.eig(cov_matrix)

In [43]:
print(eigvals)

[0.04522181 4.0696904 ]


In [44]:
print(eigvecs)

[[-0.89794091 -0.44011603]
 [ 0.44011603 -0.89794091]]


Look at the eigenvalues: 0.045 and 4.07 — wildly different sizes. That's the entire point of PCA.

The second eigenvector [-0.44, -0.898] (paired with eigenvalue 4.07) points in the direction where your data varies the most — that's the diagonal direction where x and y move together (since you built y ≈ 2x). The first eigenvector (eigenvalue 0.045) points in a direction with almost no variance — basically noise.

This is exactly what PCA does: it finds these directions, ranks them by eigenvalue (variance), and lets you keep the high-variance direction and drop the low-variance one. You'd go from 2 dimensions down to 1, keeping almost all the meaningful information (4.07 out of 4.07+0.045 ≈ 99% of total variance), and throwing away the dimension that was mostly noise.

That's the real-world payoff: hundreds of features compressed into a handful, with minimal information loss — same math, just scaled up.

You've now connected raw linear algebra to an actual ML technique you'll use in real projects. That's a strong place to pause — eigenvalues/eigenvectors plus their direct application.

#### Mean and Variance

In [46]:
data_1d = data[:, 0]  # just the x column from earlier

In [47]:
mean = np.mean(data_1d)

In [48]:
variance = np.var(data_1d)

In [49]:
std = np.std(data_1d)

In [50]:
print(f"Mean: {mean:.4f}")

Mean: -0.1038


In [51]:
print(f"Variance: {variance:.4f}")

Variance: 0.8165


In [52]:
print(f"Std Dev: {std:.4f}")

Std Dev: 0.9036
